# Phase 1: Training Pipeline — MobileNetV3-Large Fine-tuning (Task A, Balanced)

**Task**: Conifer Family Classification (Task A)

**Dataset**: `data/task-A/` — augmented dataset sau Phase 0 augmentation.
2 lop thieu so (Podocarpaceae, Taxaceae) da duoc augment tu 217/206 len 1,600 mau moi.
Train split hien co 7,264 mau (thay vi 4,487 mau imbalanced truoc do).

**A100 40GB Large-Batch Strategy**:
- Batch 256 (4× so với 64 ban đầu), sqrt-LR scaling, 5-epoch warmup + cosine
- Head LR = 2e-4, Backbone LR = 2e-5 (scaled ×2 từ 1e-4/1e-5)
- 240 epochs (tương đương ~6,720 gradient updates, so với 5,650 ở batch 64×50 epochs)
- Label smoothing 0.1, AMP

---

## Contents

1. [Setup & Root Detection](#1-setup)
2. [Colab Setup](#2-setup-colab)
3. [Install Dependencies](#3-install)
4. [Git Checkout](#4-git)
5. [Download Dataset from HuggingFace](#5-download)
6. [Verify Dataset](#6-verify)
7. [Create Dataloaders](#7-dataloaders)
8. [Create & Test Model](#8-create-model)
9. [Fine-tune MobileNetV3-Large](#9-finetune)
10. [Evaluate on Test Set](#10-evaluate)

## 1. Setup & Root Detection <a id='1-setup'></a>

Detects whether running on Colab or local machine, and sets `ROOT` accordingly.

- **Colab**: `ROOT = /content/woodbench`
- **Local**: `ROOT = <repo-root>` (auto-detected from notebook location)

Colab setup (HF login, git clone, Drive mount) is in [Cell 2](#2-setup-colab).

In [ ]:
# Colab setup: HF login, git clone, mount Drive.
from pathlib import Path

if Path("/content").exists():
    import subprocess
    import os

    # Install deps
    subprocess.run(
        ["pip", "install", "-q", "huggingface_hub"],
        capture_output=True
    )

    # -------------------------
    # Secrets / Authentication
    # -------------------------
    try:
        from google.colab import userdata
        from huggingface_hub import login

        # MUST match secret names in Colab Secrets
        HF_TOKEN = userdata.get("HF_TOKEN")
        GH_TOKEN = userdata.get("GH_TOKEN")

        # HuggingFace login
        if HF_TOKEN:
            login(token=HF_TOKEN)
            print("[HF] Authenticated")
        else:
            print("[HF] No token — public access")

        # Export GitHub token
        if GH_TOKEN:
            os.environ["GH_TOKEN"] = GH_TOKEN
            print("[Git] Token loaded")
        else:
            print("[Git] No token found")

    except Exception as e:
        print(f"[Secrets] Failed: {e}")

    # -------------------------
    # Clone repo
    # -------------------------
    repo_path = Path("/content/woodbench")

    if not repo_path.exists():
        token = os.environ.get("GH_TOKEN")

        try:
            if token:
                subprocess.run(
                    [
                        "git",
                        "clone",
                        f"https://{token}@github.com/ManifoldsTeam/woodbench.git"
                    ],
                    check=True
                )
                print("[Git] Cloned woodbench (private auth)")
            else:
                subprocess.run(
                    [
                        "git",
                        "clone",
                        "https://github.com/ManifoldsTeam/woodbench.git"
                    ],
                    check=True
                )
                print("[Git] Cloned woodbench (public)")
        except subprocess.CalledProcessError as e:
            print(f"[Git] Clone failed: {e}")

    else:
        print("[Git] woodbench already cloned")

    # -------------------------
    # Mount Google Drive
    # -------------------------
    try:
        from google.colab import drive

        if not Path("/content/drive").exists():
            drive.mount("/content/drive")
            print("[Drive] Mounted")
        else:
            print("[Drive] Already mounted")

    except Exception as e:
        print(f"[Drive] Mount failed: {e}")

else:
    print("[Local] Skipping Colab-only setup")

[HF] Authenticated
[Git] Token loaded
[Git] Cloned woodbench (private auth)
Mounted at /content/drive
[Drive] Mounted


In [ ]:
import sys
from pathlib import Path

# Root detection: Colab uses /content/woodbench, local auto-detects from notebook location.
if Path("/content").exists():
    ROOT = Path("/content/woodbench")
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

import torch
print(f"[ROOT] {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[ROOT] /content/woodbench
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 3. Install Dependencies <a id='3-install'></a>

Install project dependencies. Only runs on Colab (guarded by `/content` check).

In [ ]:
# Only run on Colab
# if Path("/content").exists():
#     get_ipython().system('pip install -q -r {ROOT}/requirements.txt')

## 4. Git Checkout <a id='4-git'></a>

Checkout the experiment branch on Colab.

In [ ]:
%cd woodbench
!git checkout feat/task-A

/content/woodbench
Branch 'feat/task-A' set up to track remote branch 'feat/task-A' from 'origin'.
Switched to a new branch 'feat/task-A'


## 5. Download Dataset from HuggingFace <a id='5-download'></a>

Downloads the augmented `data/task-A/` dataset from HuggingFace Hub.
Dataset was created in Phase 0: Podocarpaceae and Taxaceae are augmented
from 217/206 originals to 1,600 samples each, giving 7,264 train samples total.

In [ ]:
from pathlib import Path

# KEY CHANGE: Use data/task-A/ (augmented dataset) instead of data/split/ (imbalanced).
# Augmentation was done in Phase 0 notebook: notebooks/phase 0/3_task_a_augmentation.ipynb
# - Podocarpaceae: 217 originals -> 1,600 total (augmented)
# - Taxaceae:      206 originals -> 1,600 total (augmented)
# - Cupressaceae:  2,398 originals (unchanged)
# - Pinaceae:      1,666 originals (unchanged)
# Total train: 7,264 (vs 4,487 imbalanced before)

DATA_DIR = ROOT / "data" / "task-A"

# Download if missing
if not DATA_DIR.exists():
    from huggingface_hub import snapshot_download

    print("[WARN] Dataset not found locally")
    print("Downloading dataset from HuggingFace...")

    snapshot_download(
        repo_id="ny0g507h3p/wood-task-A",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )

    print("[OK] Download complete")

# Validate dataset
train_path = DATA_DIR / "train"

if train_path.exists():
    train_samples = (
        sum(1 for f in train_path.rglob("*.png")) +
        sum(1 for f in train_path.rglob("*.jpg"))
    )

    print(f"[OK] Dataset found at {DATA_DIR}")
    print(f"     Train images (approx): {train_samples}")

    if train_samples >= 7000:
        print("     Status: BALANCED dataset (~7,264 expected)")
    else:
        print("     WARNING: Expected ~7,264 train images")
        print("     This may be the old imbalanced dataset")
else:
    print(f"[ERROR] Missing train/ directory in {DATA_DIR}")

print(f"\nDATA_DIR = {DATA_DIR}")

[WARN] Dataset not found locally


Fetching ... files: 0it [00:00, ?it/s]

[OK] Download complete
[OK] Dataset found at /content/woodbench/data/task-A
     Train images (approx): 7264
     Status: BALANCED dataset (~7,264 expected)

DATA_DIR = /content/woodbench/data/task-A


## 6. Verify Dataset <a id='6-verify'></a>

Loads the train dataset and shows class distribution. With augmented data we expect:
- Cupressaceae: ~2,398 (majority, no augmentation)
- Pinaceae: ~1,666 (majority, no augmentation)
- Podocarpaceae: ~1,600 (minority, augmented from 217)
- Taxaceae: ~1,600 (minority, augmented from 206)
- Total train: ~7,264 (vs 4,487 imbalanced)

In [ ]:
from src.data import ConiferDataset

# Check structure
print("Dataset structure:")
for split in ["train", "val", "test"]:
    split_path = DATA_DIR / split
    if split_path.exists():
        families = sorted([d.name for d in split_path.iterdir() if d.is_dir()])
        print(f"  {split}/: {len(families)} families - {families}")

# Load train set and show distribution
train_ds = ConiferDataset(DATA_DIR, split="train", task="taskA")
print(f"\nTotal training samples: {len(train_ds)}")
print(f"Classes: {train_ds.classes}")
print(f"\nClass distribution:")
for family, count in train_ds.get_class_distribution().items():
    pct = count / len(train_ds) * 100
    print(f"  {family}: {count} ({pct:.1f}%)")

total = len(train_ds)
expected = 2398 + 1666 + 1600 + 1600
print(f"\nExpected total (balanced): {expected}")
print(f"Actual total: {total}")
print(f"Match: {'OK' if total == expected else 'CHECK DATA'}")
print(f"\nAugmented minority classes: Podocarpaceae={train_ds.get_class_distribution()['Podocarpaceae']}, "
      f"Taxaceae={train_ds.get_class_distribution()['Taxaceae']}")

Dataset structure:
  train/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  val/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  test/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

Total training samples: 7264
Classes: ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

Class distribution:
  Cupressaceae: 2398 (33.0%)
  Pinaceae: 1666 (22.9%)
  Podocarpaceae: 1600 (22.0%)
  Taxaceae: 1600 (22.0%)

Expected total (balanced): 7264
Actual total: 7264
Match: OK

Augmented minority classes: Podocarpaceae=1600, Taxaceae=1600


## 7. Create Dataloaders <a id='7-dataloaders'></a>

Create train/val/test DataLoaders with ImageNet normalization and online augmentation.
`use_90rotation=False` for the first run (ablation variant is in Phase 3).

In [ ]:
from src.data import get_dataloaders

BATCH_SIZE = 256  # A100 40GB: ~31GB peak (forward+backward + Adam states, AMP)
                  # batch 64→256 (×4), sqrt scaling ×2 cho LR
IMG_SIZE = 224     # MobileNetV3 native resolution
NUM_WORKERS = 4
USE_90ROTATION = False  # Ablation variant in Phase 3

train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
    use_90rotation=USE_90ROTATION,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

# Quick class balance check on first batch
unique, counts = labels.unique(return_counts=True)
print(f"\nFirst batch class distribution:")
for u, c in zip(unique, counts):
    print(f"  Class {u.item()}: {c.item()} samples")

Train batches: 28
Val batches:   4
Test batches:  4

Batch shape: torch.Size([256, 3, 224, 224])
Labels shape: torch.Size([256])

First batch class distribution:
  Class 0: 81 samples
  Class 1: 64 samples
  Class 2: 53 samples
  Class 3: 58 samples


## 8. Create & Test Model <a id='8-create-model'></a>

Creates MobileNetV3-Large with unfrozen backbone (fine-tuning mode). All parameters are trainable.
Differential learning rates will be applied in the optimizer (next cell).

In [ ]:
from src.models import create_model, BACKBONE_NAMES

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Check available backbones
print("\nAvailable backbones:")
for name, timm_name in BACKBONE_NAMES.items():
    print(f"  {name}: {timm_name}")

# Create model — unfrozen for fine-tuning (all param groups trainable)
model = create_model("mobilenetv3_large", num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Test forward pass with small batch (avoid OOM)
test_images = images[:16]  # Only 16 samples to verify model works
outputs = model(test_images.to(device))
print(f"\nOutput shape: {outputs.shape}  (expected: (16, 4))")
del test_images, outputs
torch.cuda.empty_cache()

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal params:    {total_params / 1e6:.2f}M")
print(f"Trainable:       {trainable_params / 1e6:.2f}M")
print(f"Frozen:          {frozen_params / 1e6:.2f}M")
print(f"Fine-tuning mode: {'UNFROZEN' if frozen_params == 0 else 'PARTIALLY FROZEN'}")

Using device: cuda

Available backbones:
  resnet50: resnet50
  efficientnet_b3: tf_efficientnet_b3
  mobilenetv3_large: mobilenetv3_large_100
  convnext_tiny: convnext_tiny
  swin_tiny: swin_tiny_patch4_window7_224

Output shape: torch.Size([16, 4])  (expected: (16, 4))

Total params:    4.21M
Trainable:       4.21M
Frozen:          0.00M
Fine-tuning mode: UNFROZEN


## 9. Fine-tune MobileNetV3-Large <a id='9-finetune'></a>

Large-batch strategy (Goyal et al., "Accurate, Large Minibatch SGD", 2018):
- Head LR = 2e-4, Backbone LR = 2e-5 — sqrt-LR scaling (×2) so với batch 64×1e-4/1e-5
- 5-epoch linear warmup → cosine annealing
- 240 epochs (7,264 samples / 256 batch = 28 steps/epoch → 6,720 total updates)
- Label smoothing 0.1, AMP

Save all checkpoints (`save_best_only=False`) to track convergence curve.

In [ ]:
from pathlib import Path

# Output directory: Google Drive on Colab, local experiments/ on desktop.
# Note: using "mobilenetv3_large_balanced" to distinguish from old runs.
if Path("/content/drive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = ROOT / "experiments"

EXP_DIR = DRIVE_ROOT / "woodbench_experiments" / "task-A" / "mobilenetv3_large_balanced"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Output] {EXP_DIR}")

[Output] /content/drive/MyDrive/woodbench_experiments/task-A/mobilenetv3_large_balanced


In [ ]:
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

set_seed(42)

# Re-create model fresh from pretrained weights
model = create_model("mobilenetv3_large", num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Large-batch: sqrt-LR scaling ×2, warmup 5 epochs
HEAD_LR = 2e-4    # scaled from 1e-4 (sqrt(256/64) = 2.0)
BACKBONE_LR = 2e-5  # scaled from 1e-5
optimizer = create_optimizer(
    model,
    lr=HEAD_LR,
    weight_decay=1e-4,
    backbone_lr=BACKBONE_LR,
)

print("Optimizer param groups:")
for pg in optimizer.param_groups:
    print(f"  [{pg['name']}] lr={pg['lr']:.2e}, {len(pg['params'])} tensors")

scheduler = create_scheduler(optimizer, epochs=240, scheduler_type="cosine", warmup_epochs=5)
criterion = create_loss_function(loss_type="label_smoothing", label_smoothing=0.1)

engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=True,
)

print("\nStarting fine-tuning (240 epochs, large-batch strategy)...")
print(f"Train samples:   {len(train_ds)}")
print(f"Steps/epoch:    {len(train_loader)}")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Head LR:        {HEAD_LR}  (scaled ×2 from 1e-4)")
print(f"Backbone LR:    {BACKBONE_LR}  (scaled ×2 from 1e-5)")
print(f"Warmup epochs:  5  (linear)")
print(f"Total epochs:   240")
print(f"Scheduler:      cosine with warmup")
print(f"Label smooth:   0.1")
print(f"AMP:            enabled\n")

results = engine.fit(
    epochs=240,
    output_dir=EXP_DIR,
    name="mobilenetv3_large",
    save_best_only=False,
    verbose=True,
)

print(f"\nBest val accuracy: {results['best_val_acc']:.2f}%")
print(f"Final train acc:  {results['final_train_acc']:.2f}%")
print(f"Final val acc:    {results['final_val_acc']:.2f}%")

Optimizer param groups:
  [head] lr=2.00e-04, 4 tensors
  [backbone] lr=2.00e-05, 170 tensors

Starting fine-tuning (240 epochs, large-batch strategy)...
Train samples:   7264
Steps/epoch:    28
Batch size:     256
Head LR:        0.0002  (scaled ×2 from 1e-4)
Backbone LR:    2e-05  (scaled ×2 from 1e-5)
Warmup epochs:  5  (linear)
Total epochs:   240
Scheduler:      cosine with warmup
Label smooth:   0.1
AMP:            enabled



Epoch 1/240 | Train Loss: 1.3952, Acc: 26.02% | Val Loss: 1.3981, Acc: 19.69% | LR: 4.00e-05[Best]
  ↳ New best! Saved to mobilenetv3_large_best.pth (Val Acc: 19.69%)


Epoch 2/240 | Train Loss: 1.2824, Acc: 48.23% | Val Loss: 1.1226, Acc: 66.91% | LR: 8.00e-05[Best]
  ↳ New best! Saved to mobilenetv3_large_best.pth (Val Acc: 66.91%)


Epoch 3/240 | Train Loss: 1.0276, Acc: 71.88% | Val Loss: 0.8621, Acc: 78.95% | LR: 1.20e-04[Best]
  ↳ New best! Saved to mobilenetv3_large_best.pth (Val Acc: 78.95%)


Epoch 4/240 | Train Loss: 0.7451, Acc: 84.65% | Val Loss: 0.6549, Acc: 86.28% | LR: 1.60e-04[Best]
  ↳ New best! Saved to mobilenetv3_large_best.pth (Val Acc: 86.28%)


Epoch 5/240 | Train Loss: 0.5695, Acc: 91.71% | Val Loss: 0.5412, Acc: 92.77% | LR: 2.00e-04[Best]
  ↳ New best! Saved to mobilenetv3_large_best.pth (Val Acc: 92.77%)


... (33 outputs left out)

Epoch 39/240 | Train Loss: 0.3644, Acc: 99.92% | Val Loss: 0.3728, Acc: 99.58% | LR: 1.90e-04


Epoch 40/240 | Train Loss: 0.3641, Acc: 99.93% | Val Loss: 0.3711, Acc: 99.58% | LR: 1.89e-04


Epoch 41/240 | Train Loss: 0.3633, Acc: 99.96% | Val Loss: 0.3707, Acc: 99.58% | LR: 1.89e-04


KeyboardInterrupt: 

In [ ]:
# List all checkpoints saved during training
checkpoint_files = sorted(EXP_DIR.glob("mobilenetv3_large_*.pth"))
print(f"Total checkpoints: {len(checkpoint_files)}")
if checkpoint_files:
    print(f"First:  {checkpoint_files[0].name}")
    print(f"Last:   {checkpoint_files[-1].name}")
print(f"Best:   {EXP_DIR / 'mobilenetv3_large_best.pth'}")
print(f"All:    {len(checkpoint_files)} files")

Total checkpoints: 42
First:  mobilenetv3_large_best.pth
Last:   mobilenetv3_large_epoch9.pth
Best:   /content/drive/MyDrive/woodbench_experiments/task-A/mobilenetv3_large_balanced/mobilenetv3_large_best.pth
All:    42 files


## 10. Evaluate on Test Set <a id='10-evaluate'></a>

Loads the best checkpoint from the balanced-data training run and evaluates on the held-out test set (which was NOT augmented — contains original images only).

In [ ]:
from src.eval import compute_metrics, format_metrics, save_confusion_matrix

CLASS_NAMES = ["Cupressaceae", "Pinaceae", "Podocarpaceae", "Taxaceae"]

# Load the best checkpoint from the balanced-data training run
best_checkpoint_path = EXP_DIR / "mobilenetv3_large_best.pth"

if best_checkpoint_path.exists():
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_epoch = checkpoint.get("epoch", 0) + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    print(f"Loaded best checkpoint: epoch {best_epoch}, val_acc={best_val_acc:.2f}%")
else:
    # Fallback: find the epoch with highest validation accuracy from training history
    print(f"Best checkpoint not found at {best_checkpoint_path}")
    print("Finding best epoch from all checkpoints...")

# Evaluate on test set
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=4,
    class_names=CLASS_NAMES,
)

print(format_metrics(metrics, CLASS_NAMES))

# Save confusion matrix
save_confusion_matrix(
    cm=metrics["confusion_matrix"],
    class_names=CLASS_NAMES,
    output_path=EXP_DIR / "confusion_matrix_balanced.png",
    title="Confusion Matrix — MobileNetV3-Large Fine-tune (Balanced Data, Test Set)",
)
print(f"\nConfusion matrix saved to {EXP_DIR / 'confusion_matrix_balanced.png'}")

# Per-class accuracy breakdown
print("\nPer-class accuracy (from confusion matrix):")
cm = metrics["confusion_matrix"]
for i, name in enumerate(CLASS_NAMES):
    class_correct = cm[i, i]
    class_total = cm[i, :].sum()
    class_acc = class_correct / class_total * 100 if class_total > 0 else 0
    print(f"  {name}: {class_correct}/{class_total} = {class_acc:.1f}%")

Loaded best checkpoint: epoch 23, val_acc=99.58%
EVALUATION RESULTS
Accuracy:      99.08%
Precision:     97.64%
Recall:        99.02%
F1-macro:      98.31%

Per-Class F1 Scores:
  Cupressaceae        : 99.24%
  Pinaceae            : 99.31%
  Podocarpaceae       : 96.91%
  Taxaceae            : 97.78%

Confusion matrix saved to /content/drive/MyDrive/woodbench_experiments/task-A/mobilenetv3_large_balanced/confusion_matrix_balanced.png

Per-class accuracy (from confusion matrix):
  Cupressaceae: 519/525 = 98.9%
  Pinaceae: 362/364 = 99.5%
  Podocarpaceae: 47/47 = 100.0%
  Taxaceae: 44/45 = 97.8%
